# Pidgin camerounais → anglais : ajustement de NITHUB-AI/marian-mt-bbc-pcm-en (Kaggle)

**Avant :**
1. Ajouter le jeu de données `kama-pcm-en-data` (dossier `training/pidgin_pcm_en/data` : train.jsonl, dev.jsonl, test_soreva.jsonl), en **Private**. Données dérivées de MT560 (CC BY 4.0).
2. **Add Input** → ce jeu de données. **Settings** : GPU T4 x2 (un seul est utilisé), **Internet = On** (téléchargement du modèle de départ).
3. Essai : `SMOKE = True`, *Run All*. Puis `SMOKE = False`, *Save Version → Save & Run All (Commit)*.

Sortie : `kama_pcm_en_ct2.zip` (modèle CTranslate2 int8, prêt pour le serveur) et `evaluation.txt` (chrF base contre ajusté sur SOREVA). Rien n'est mis dans le dépôt Git.


In [ ]:
import sys, os, glob, shutil
import torch
assert torch.cuda.is_available(), 'Pas de GPU : Settings → Accelerator = GPU T4 x2'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('Python', sys.version.split()[0], '|', torch.cuda.get_device_name(0))
SMOKE = True
TMP = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
SORTIE_FINALE = '/kaggle/working'
print('mode', 'ESSAI' if SMOKE else 'ENTRAÎNEMENT COMPLET')


In [ ]:
# Données : le dossier du jeu de données ajouté au carnet (quel que soit son nom sous /kaggle/input)
trouves = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
assert trouves, 'Données introuvables : Add Input → kama-pcm-en-data'
DONNEES = os.path.dirname(trouves[0])
print('données :', DONNEES, '|', sum(1 for _ in open(trouves[0], encoding='utf-8')), 'paires d\'entraînement')


In [ ]:
!pip install -q 'transformers>=4.45,<5' sentencepiece sacremoses sacrebleu accelerate ctranslate2


In [ ]:
# Le code d'entraînement : copié depuis le dépôt (le jeu de données ne contient que les données).
# Si ce carnet est importé seul, ajouter aussi le dossier training/pidgin_pcm_en comme jeu de données (Add Input) ;
# sinon, coller le contenu de entrainer.py et evaluer.py dans des cellules.
codes = glob.glob('/kaggle/input/**/entrainer.py', recursive=True)
assert codes, 'entrainer.py introuvable : ajoute le dossier training/pidgin_pcm_en au carnet (Add Input)'
CODE = os.path.dirname(codes[0])
SORTIE = f'{TMP}/sortie_pcm_en'
shutil.rmtree(SORTIE, ignore_errors=True)
epoques = 1 if SMOKE else 3


In [ ]:
!python {CODE}/entrainer.py --donnees {DONNEES} --sortie {SORTIE} --epoques {epoques} {'--smoke' if SMOKE else ''}


In [ ]:
# Évaluation : base contre ajusté sur les 77 phrases SOREVA (pidgin → anglais). Le résultat est écrit en fichier.
import subprocess
r = subprocess.run(['python', f'{CODE}/evaluer.py', '--test', f'{DONNEES}/test_soreva.jsonl', '--modele', f'{SORTIE}/final'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-800:] if r.returncode else '')
open(f'{SORTIE_FINALE}/evaluation.txt', 'w', encoding='utf-8').write(r.stdout)


In [ ]:
# Conversion CTranslate2 int8 (format du serveur) et archive. Le serveur n'attend que ce dossier.
!ct2-transformers-converter --model {SORTIE}/final --output_dir {TMP}/ct2_pcm_en --quantization int8 --force
!cp {SORTIE}/final/source.spm {SORTIE}/final/target.spm {TMP}/ct2_pcm_en/ 2>/dev/null || cp {SORTIE}/final/*.spm {TMP}/ct2_pcm_en/
shutil.make_archive(f'{SORTIE_FINALE}/kama_pcm_en_ct2', 'zip', TMP, 'ct2_pcm_en')
print('archive :', os.path.getsize(f'{SORTIE_FINALE}/kama_pcm_en_ct2.zip') // 1_000_000, 'Mo')
